# IQ GLM-5.3 — Automatic Colab runtime preflight

Detects the **actual attached accelerator** using JAX (`device.platform == "tpu"`) or PyTorch CUDA, then runs a tiny TPU/GPU smoke check or CPU-only metadata checks.

- **No model weight downloads, no fine-tuning, no full-256 inference.**
- No `pip install`, no runtime provisioning or purchase. The notebook **does not change** your Colab runtime.
- Current official Colab CLI accelerator requests are `--tpu v5e1`, `--tpu v6e1` and `--gpu T4|L4|G4|A100|H100`. The notebook does **not** guess from those labels: it identifies the physical attached hardware.

Optional: if the original HF bucket is already mounted, set `IQ_CHECKPOINT` and `IQ_DEVICE_MAP` below to run the full metadata-only preflight. TPU does not run CUDA Mamba-3 MIMO kernels.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import subprocess, sys, json

# Optional: set to existing remote HF bucket MOUNTS, not model download paths.
IQ_CHECKPOINT = ""
IQ_DEVICE_MAP = ""
IQ_TOKENIZER_DIR = ""
RUN_TINY_SMOKE = True

# Fetch only small, reviewable validation Python scripts from IQ's draft PR.
source = "https://raw.githubusercontent.com/Harqer/IQ/fix/iq-full256-mapped-inference-preflight-20261114/scripts/"
script_dir = Path("/tmp/iq_colab_runtime_checks")
script_dir.mkdir(parents=True, exist_ok=True)
names = ["iq_colab_runtime_probe.py"]
if IQ_CHECKPOINT and IQ_DEVICE_MAP:
    names += ["check_iq_device_map.py"]
for name in names:
    with urlopen(source + name, timeout=30) as response:
        (script_dir / name).write_bytes(response.read())

command = [sys.executable, str(script_dir / "iq_colab_runtime_probe.py")]
if RUN_TINY_SMOKE:
    command.append("--smoke")
if IQ_DEVICE_MAP:
    command += ["--device-map", IQ_DEVICE_MAP]
if IQ_CHECKPOINT:
    command += ["--checkpoint", IQ_CHECKPOINT]
if IQ_TOKENIZER_DIR:
    command += ["--tokenizer-dir", IQ_TOKENIZER_DIR]

result = subprocess.run(command, check=False)
print("Preflight exit status:", result.returncode)
if result.returncode:
    raise RuntimeError("IQ hardware preflight reported a failure; review diagnostics above.")
